In [1]:
import json
import pandas as pd
from collections import defaultdict
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

DATA_DIR = Path("../data")  # if notebook is inside notebooks folder

INPUT_FILE = DATA_DIR / "all_files_extracted_data.json"
OUTPUT_FILE = DATA_DIR / "processed_workforce_data.json"

# ============================================================
# LOAD EXTRACTED JSON
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

workbook = data["Wohlig Active Employee Data.xlsx"]

active_data = workbook["Active"]

# ============================================================
# ACTIVE DATAFRAME
# ============================================================

active_df = pd.DataFrame(active_data)

active_df.columns = [
    "sr_no",
    "employee_name",
    "doj",
    "role",
    "project",
    "location",
    "project_start",
    "project_end",
    "manager"
]

# Remove header row
active_df = active_df[
    active_df["employee_name"] != "Active Employee"
].reset_index(drop=True)

# ============================================================
# BUILD EMPLOYEE STRUCTURE
# ============================================================

employees = {}
current_employee = None

for _, row in active_df.iterrows():

    employee_name = row["employee_name"]

    # Main Employee Row
    if pd.notna(employee_name):

        current_employee = str(employee_name).strip()

        employees[current_employee] = {
            "name": current_employee,
            "role": str(row["role"]).strip(),
            "manager": str(row["manager"]).strip(),
            "allocations": []
        }

        if pd.notna(row["project"]):
            employees[current_employee]["allocations"].append(
                str(row["project"]).strip()
            )

    # Continuation Row
    else:

        if current_employee and pd.notna(row["project"]):
            employees[current_employee]["allocations"].append(
                str(row["project"]).strip()
            )

# ============================================================
# MULTI ALLOCATED
# ============================================================

multi_allocated = [
    emp
    for emp in employees.values()
    if len(emp["allocations"]) > 1
]
multi_allocated_employee_list = [
    emp["name"]
    for emp in multi_allocated
]

# ============================================================
# UNALLOCATED
# ============================================================

unallocated = []

for emp in employees.values():

    allocations = [
        str(a).strip().lower()
        for a in emp["allocations"]
    ]

    if (
        len(allocations) == 0
        or allocations == ["no task"]
        or allocations == [""]
    ):
        unallocated.append(emp)


# ============================================================
# BUILD CATEGORY MAPPING FROM SHEETS
# ============================================================

classification_map = {}
closed_projects = []
completed_projects = []

# -------------------------
# PROJECT SHEET
# -------------------------

for row in workbook["Project"]:

    project_name = row.get("Project Name")
    status = str(row.get("Status", "")).strip().lower()

    if not project_name:
        continue

    project_name = str(project_name).strip()

    if status == "closed":
        closed_projects.append(project_name)

    elif status == "done":
        completed_projects.append(project_name)

    else:
        classification_map[project_name.lower()] = "Project"

# -------------------------
# RETAINER SHEET
# -------------------------

for row in workbook["Retainers"]:

    project_name = row.get("Project Name")
    status = str(row.get("Status", "")).strip().lower()

    if not project_name:
        continue

    project_name = str(project_name).strip()

    if status != "closed":
        classification_map[project_name.lower()] = "Retainer"

# -------------------------
# INTERNAL SHEET
# -------------------------

for row in workbook["Internal"]:

    project_name = row.get("Project Name")
    status = str(row.get("Status", "")).strip().lower()

    if not project_name:
        continue

    project_name = str(project_name).strip()

    if status == "closed":

        closed_projects.append(project_name)

    else:

        classification_map[project_name.lower()] = "Internal"


# ============================================================
# CLOSED PROJECT LOOKUP
# ============================================================

closed_projects_lower = {
    project.lower()
    for project in closed_projects
}

# ============================================================
# CATEGORY COUNTS
# ============================================================

project_count = 0
retainer_count = 0
internal_count = 0

for emp in employees.values():

    allocations = [
        str(a).strip().lower()
        for a in emp["allocations"]
    ]

    if allocations == ["no task"] or len(allocations) == 0:
        continue

    is_project = False
    is_retainer = False
    is_internal = False

    for allocation in allocations:

        category = classification_map.get(allocation)

        if category == "Project":
            is_project = True

        elif category == "Retainer":
            is_retainer = True

        elif category == "Internal":
            is_internal = True

    if is_project:
        project_count += 1

    if is_retainer:
        retainer_count += 1

    if is_internal:
        internal_count += 1

# ============================================================
# WORKFORCE OVERVIEW
# ============================================================


workforce_overview = {
    "active_employees": len(employees),

    "unallocated_employees": len(unallocated),

    "multi_allocated_employees": len(multi_allocated),
    "multi_allocated_employee_names": multi_allocated_employee_list,

    "project_employees": project_count,
    "retainer_employees": retainer_count,
    "internal_employees": internal_count,

    "completed_projects": len(completed_projects),
    "completed_project_names": completed_projects,

    "closed_projects": len(closed_projects),
    "closed_project_names": closed_projects
}
# ============================================================
# UNALLOCATED EMPLOYEE LIST
# ============================================================

unallocated_employee_list = []

for emp in unallocated:

    unallocated_employee_list.append({
        "name": emp["name"],
        "current_role": emp["role"],
        "reporting_to": emp["manager"]
    })

# ============================================================
# PROJECT ALLOCATION SUMMARY
# ============================================================

allocation_summary = defaultdict(list)

for emp in employees.values():

    for allocation in emp["allocations"]:

        allocation_summary[allocation].append(
            emp["name"]
        )

project_allocation_summary = []

NORMALIZATION = {
    "apollo hispital": "apollo hospital"
}

closed_projects_lower = {
    p.strip().lower()
    for p in closed_projects
}

completed_projects_lower = {
    p.strip().lower()
    for p in completed_projects
}

for project_name, employee_names in allocation_summary.items():

    normalized_name = NORMALIZATION.get(
        project_name.strip().lower(),
        project_name.strip().lower()
    )

    # Skip No Task
    if normalized_name == "no task":
        continue

    # Skip Completed Projects
    if normalized_name in completed_projects_lower:
        continue

    # Skip Closed Projects
    if normalized_name in closed_projects_lower:
        continue

    project_allocation_summary.append({
        "project_name": project_name,
        "employee_count": len(employee_names),
        "employees": sorted(employee_names)
    })

    project_allocation_summary = sorted(
    project_allocation_summary,
    key=lambda x: x["employee_count"],
    reverse=True
)

# ============================================================
# FINAL OUTPUT
# ============================================================

processed_data = {
    "workforce_overview": workforce_overview,
    "unallocated_employee_list": unallocated_employee_list,
    "project_allocation_summary": project_allocation_summary
}

# ============================================================
# SAVE JSON
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        processed_data,
        f,
        indent=4,
        ensure_ascii=False
    )

# ============================================================
# DISPLAY
# ============================================================

print("\n========== WORKFORCE OVERVIEW ==========")
print(json.dumps(workforce_overview, indent=4))

print("\n========== UNALLOCATED EMPLOYEES ==========")

for emp in unallocated_employee_list:
    print(emp)

print("\n========== PROJECT ALLOCATION SUMMARY ==========")

for project in project_allocation_summary:
    print(
        f"{project['project_name']} -> "
        f"{project['employee_count']} employees"
    )

print(f"\nSaved: {OUTPUT_FILE}")


========== WORKFORCE OVERVIEW ==========
{
    "active_employees": 31,
    "unallocated_employees": 6,
    "multi_allocated_employees": 2,
    "multi_allocated_employee_names": [
        "Mustafa Mohammed",
        "Agam Shah"
    ],
    "project_employees": 2,
    "retainer_employees": 13,
    "internal_employees": 9,
    "completed_projects": 1,
    "completed_project_names": [
        "Apollo Hospital"
    ],
    "closed_projects": 1,
    "closed_project_names": [
        "Pocket Studio"
    ]
}

========== UNALLOCATED EMPLOYEES ==========
{'name': 'Samay Gada', 'current_role': 'Tech Lead', 'reporting_to': 'Chintan Shah'}
{'name': 'Siddesh Kale', 'current_role': 'Software Developer', 'reporting_to': 'Dhruv Gansinghani'}
{'name': 'Nishant Patil', 'current_role': 'Software Developer', 'reporting_to': 'Dhruv Gansinghani'}
{'name': 'Chintan Limbad', 'current_role': 'Data Engineer - Fresher', 'reporting_to': 'Rajvee Gala'}
{'name': 'Praveen Kumar', 'current_role': 'Gen AI - Fresher', 'r